In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upgrade pip to handle modern packages
!pip install --upgrade pip

# Install Nerfstudio (Letting it auto-detect the best torch/numpy versions)
!pip install nerfstudio

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get update
!apt-get install -y colmap ffmpeg

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,510 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backp

In [ ]:
# 3. Process video with DOWN-SCALING to prevent crash
# We use a new folder 'processed_data_v3'
!ns-process-data video \
    --data /content/drive/MyDrive/Project2_NeRF/books.mp4 \
    --output-dir /content/drive/MyDrive/Project2_NeRF/processed_data_v3 \
    --num-frames-target 50 \
    --downscale-factor 2 \
    --verbose

╭─ Unrecognized options ────────────────────────╮
│ Unrecognized options: --downscale-factor      │
│ ───────────────────────────────────────────── │
│ For full helptext, run ns-process-data --help │
╰───────────────────────────────────────────────╯


In [ ]:
# Train the NeRF model
# We use the 'nerfacto' method
!ns-train nerfacto \
    --data /content/drive/MyDrive/Project2_NeRF/processed_data \
    --output-dir /content/drive/MyDrive/Project2_NeRF/outputs \
    --experiment-name my-nerf-experiment \
    --max-num-iterations 5000 \
    --vis wandb

2025-12-07 15:37:07.967555: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765121827.988000    5929 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765121827.994072    5929 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765121828.009304    5929 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765121828.009328    5929 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765121828.009333    5929 computation_placer.cc:177] computation placer alr

In [ ]:
# Train the Gaussian Splatting model
# We use the 'splatfacto' method
!ns-train splatfacto \
    --data /content/drive/MyDrive/Project2_NeRF/processed_data \
    --output-dir /content/drive/MyDrive/Project2_NeRF/outputs \
    --experiment-name my-splat-experiment \
    --max-num-iterations 5000 \
    --vis wandb

2025-12-07 15:38:02.269084: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765121882.289235    6168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765121882.295274    6168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765121882.310506    6168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765121882.310534    6168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765121882.310538    6168 computation_placer.cc:177] computation placer alr

In [ ]:
import os
from glob import glob

# Find the config files automatically
base_dir = "/content/drive/MyDrive/Project2_NeRF/outputs"

# Find NeRF config
nerf_configs = glob(f"{base_dir}/my-nerf-experiment/nerfacto/*/config.yml")
nerf_config_path = nerf_configs[-1] if nerf_configs else "NOT FOUND"

# Find Splat config
splat_configs = glob(f"{base_dir}/my-splat-experiment/splatfacto/*/config.yml")
splat_config_path = splat_configs[-1] if splat_configs else "NOT FOUND"

print(f"NeRF Config: {nerf_config_path}")
print(f"Splat Config: {splat_config_path}")

NeRF Config: /content/drive/MyDrive/Project2_NeRF/outputs/my-nerf-experiment/nerfacto/2025-12-07_153728/config.yml
Splat Config: /content/drive/MyDrive/Project2_NeRF/outputs/my-splat-experiment/splatfacto/2025-12-07_153816/config.yml


In [ ]:
# Render NeRF Video (Spiral Path)
!ns-render interpolate \
    --load-config "{nerf_config_path}" \
    --output-path /content/drive/MyDrive/Project2_NeRF/nerf_result.mp4 \
    --frame-rate 24

2025-12-07 15:40:38.077217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765122038.096814    6827 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765122038.102794    6827 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765122038.117842    6827 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765122038.117866    6827 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765122038.117870    6827 computation_placer.cc:177] computation placer alr

In [ ]:
# Render Splat Video (Spiral Path)
!ns-render interpolate \
    --load-config "{splat_config_path}" \
    --output-path /content/drive/MyDrive/Project2_NeRF/splat_result.mp4 \
    --frame-rate 24

2025-12-07 15:42:21.345857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765122141.366407    7274 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765122141.372400    7274 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765122141.387180    7274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765122141.387215    7274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765122141.387226    7274 computation_placer.cc:177] computation placer alr

In [ ]:
!find /content/drive/MyDrive/Project2_NeRF -name "transforms.json"

In [ ]:
!ls -lh /content/drive/MyDrive/Project2_NeRF/

total 241M
-rw------- 1 root root 241M Dec  7 15:13 books.mp4
drwx------ 4 root root 4.0K Dec  7 15:38 outputs
drwx------ 7 root root 4.0K Dec  7 15:34 processed_data


In [ ]:
# 1. Install COLMAP again just to be safe (it sometimes disappears if runtime resets)
!apt-get install -y colmap ffmpeg

# 2. Run processing with a NEW output folder name
!ns-process-data video \
    --data /content/drive/MyDrive/Project2_NeRF/books.mp4 \
    --output-dir /content/drive/MyDrive/Project2_NeRF/processed_data_v2 \
    --num-frames-target 50 \
    --verbose

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
colmap is already the newest version (3.7-2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 66 not upgraded.
Number of frames in video: 6921
Extracting 51 frames in evenly spaced intervals
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-li

In [ ]:
import os

# Define the folder we just tried to create
folder = "/content/drive/MyDrive/Project2_NeRF/processed_data_v2/images"

# Check if the folder exists and has images
if os.path.exists(folder):
    images = os.listdir(folder)
    print(f"STATUS: Found {len(images)} images extracted.")
    if len(images) > 0:
        print("DIAGNOSIS: Frame extraction worked. The crash happened during COLMAP (Math).")
    else:
        print("DIAGNOSIS: The folder exists but is empty. Frame extraction failed.")
else:
    print("STATUS: The 'images' folder does not exist.")
    print("DIAGNOSIS: Frame extraction failed completely. It couldn't even start.")

STATUS: Found 51 images extracted.
DIAGNOSIS: Frame extraction worked. The crash happened during COLMAP (Math).
